# LLM-105 Allegro Potential NVE Demonstration

This notebook builds LAMMPS with the official `pair_nequip_allegro` interface and runs a short microcanonical (NVE) molecular-dynamics demonstration for a 304-atom LLM-105 crystal using a fine-tuned Allegro potential.

The demonstration is intended to let a manuscript reviewer verify that the released model loads correctly, produces finite energies and forces, and sustains a short NVE trajectory. It is not a reproduction of the full training workflow or the complete production trajectory.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dantasqu/llm105-allegro-potentials/blob/main/notebooks/LLM105_Allegro_NVE_Colab.ipynb)

## Status and provenance

This is an initial Colab implementation. The software versions are pinned from a known working HPC environment and the LAMMPS installation follows the official upstream patching procedure. The direct HPC build did not preserve the exact revision of the copied pair-style files, so this notebook pins a nearby verified upstream revision that must still pass an end-to-end Colab test.

| Component | Pinned value | Basis |
| --- | --- | --- |
| PyTorch | 2.7.0 CUDA 12.6 | Known working runtime |
| NequIP | 0.17.1 | Known working direct build |
| Allegro | 0.8.1 | Known working direct build |
| e3nn | 0.5.9 | Known working direct build |
| LAMMPS | `5ea3b58ad8d72ddc1b50c102033578181d34bbbd` | Verified HPC source commit |
| pair_nequip_allegro | `2e19360b2639d960fb59223c5260eb87d0fbf273` | Verified upstream revision from May 2026 |

Official references: [Allegro LAMMPS integration](https://nequip.readthedocs.io/projects/allegro/en/latest/guide/lammps.html) and [pair_nequip_allegro](https://github.com/mir-group/pair_nequip_allegro).

## 1. Confirm a GPU runtime

In Colab, select **Runtime → Change runtime type → GPU** before continuing. A T4, L4, or A100 should have ample memory for the 304-atom demonstration.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU was detected. Enable a GPU runtime in Colab.")

subprocess.run(["nvidia-smi"], check=True)

## 2. Install the pinned Python and build dependencies

The deployed model is a self-contained TorchScript archive, but LAMMPS must be compiled and linked against a compatible PyTorch installation. This cell installs the versions observed in the known working direct build.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
     "torch==2.7.0", "--index-url", "https://download.pytorch.org/whl/cu126"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
     "nequip==0.17.1", "nequip-allegro==0.8.1", "e3nn==0.5.9",
     "cmake==3.27.7", "ninja", "requests", "pandas", "matplotlib"],
    check=True,
)
print("Pinned dependencies installed.")

In [ ]:
from importlib.metadata import version
import torch

print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
print("NequIP:", version("nequip"))
print("Allegro:", version("nequip-allegro"))
print("e3nn:", version("e3nn"))

assert torch.cuda.is_available(), "The CUDA-enabled PyTorch runtime cannot see a GPU."

## 3. Build LAMMPS with the upstream Allegro pair style

This first draft mirrors the verified direct build by using the non-Kokkos `pair_style allegro` path. MPI is disabled because Colab uses a single process; OpenMP and the LAMMPS MOLECULE package are enabled. The TorchScript model can still execute on CUDA through PyTorch.

The build may take 20–60 minutes on a fresh Colab runtime. The cell is designed to reuse a completed source checkout and build directory when rerun.

In [ ]:
%%bash
set -euo pipefail

WORK_DIR=/content/llm105_allegro_nve
LAMMPS_DIR=${WORK_DIR}/lammps
PAIR_DIR=${WORK_DIR}/pair_nequip_allegro
LAMMPS_COMMIT=5ea3b58ad8d72ddc1b50c102033578181d34bbbd
PAIR_COMMIT=2e19360b2639d960fb59223c5260eb87d0fbf273

mkdir -p "${WORK_DIR}"

if [[ ! -d "${LAMMPS_DIR}/.git" ]]; then
  git init "${LAMMPS_DIR}"
  git -C "${LAMMPS_DIR}" remote add origin https://github.com/lammps/lammps.git
  git -C "${LAMMPS_DIR}" fetch --depth 1 origin "${LAMMPS_COMMIT}"
  git -C "${LAMMPS_DIR}" checkout --detach FETCH_HEAD
fi

if [[ ! -d "${PAIR_DIR}/.git" ]]; then
  git init "${PAIR_DIR}"
  git -C "${PAIR_DIR}" remote add origin https://github.com/mir-group/pair_nequip_allegro.git
  git -C "${PAIR_DIR}" fetch --depth 1 origin "${PAIR_COMMIT}"
  git -C "${PAIR_DIR}" checkout --detach FETCH_HEAD
fi

test "$(git -C "${LAMMPS_DIR}" rev-parse HEAD)" = "${LAMMPS_COMMIT}"
test "$(git -C "${PAIR_DIR}" rev-parse HEAD)" = "${PAIR_COMMIT}"

if ! grep -q 'find_package(Torch REQUIRED)' "${LAMMPS_DIR}/cmake/CMakeLists.txt"; then
  (cd "${PAIR_DIR}" && ./patch_lammps.sh "${LAMMPS_DIR}")
else
  echo "LAMMPS source is already patched for PyTorch."
fi

TORCH_CMAKE_PREFIX=$(python -c 'import torch; print(torch.utils.cmake_prefix_path)')

cmake -S "${LAMMPS_DIR}/cmake" -B "${LAMMPS_DIR}/build" -G Ninja \
  -D CMAKE_BUILD_TYPE=Release \
  -D CMAKE_CXX_STANDARD=17 \
  -D CMAKE_PREFIX_PATH="${TORCH_CMAKE_PREFIX}" \
  -D CMAKE_INSTALL_PREFIX="${WORK_DIR}/lammps-install" \
  -D BUILD_MPI=OFF \
  -D BUILD_OMP=ON \
  -D PKG_OPENMP=ON \
  -D PKG_MOLECULE=ON \
  -D PKG_KOKKOS=OFF \
  -D NEQUIP_AOT_COMPILE=OFF \
  -D MKL_INCLUDE_DIR=/tmp

cmake --build "${LAMMPS_DIR}/build" --parallel 2
test -x "${LAMMPS_DIR}/build/lmp"
echo "LAMMPS executable: ${LAMMPS_DIR}/build/lmp"

In [ ]:
import glob
import os
from pathlib import Path
import site
import subprocess
import torch

WORK_DIR = Path("/content/llm105_allegro_nve")
LMP = WORK_DIR / "lammps/build/lmp"
torch_lib = Path(torch.__file__).resolve().parent / "lib"
nvidia_libs = []
for base in site.getsitepackages():
    nvidia_libs.extend(glob.glob(str(Path(base) / "nvidia/*/lib")))
runtime_env = os.environ.copy()
runtime_env["LD_LIBRARY_PATH"] = ":".join(
    [str(torch_lib), *nvidia_libs, runtime_env.get("LD_LIBRARY_PATH", "")]
)
runtime_env["OMP_NUM_THREADS"] = "2"

help_result = subprocess.run(
    [str(LMP), "-h"], capture_output=True, text=True, env=runtime_env, check=True
)
matching = [line for line in help_result.stdout.splitlines() if "allegro" in line.lower() or "nequip" in line.lower()]
print("\n".join(matching))
assert any("allegro" in line.lower() for line in matching), "LAMMPS was built without the Allegro pair style."

## 4. Download the selected model and NVE inputs

The repository is currently private. In Colab, add a secret named `GITHUB_TOKEN` containing a fine-grained GitHub token with **read-only Contents access** to `dantasqu/llm105-allegro-potentials`. The token is sent only as an HTTPS authorization header and is never printed.

Only one model is downloaded. The full model repository is not cloned. The NVE input and unit-cell structure are downloaded from `examples/nve/`. The archived input names its original HPC checkpoint; the preparation cell below explicitly replaces that path with the selected displayed model D / OMC25 checkpoint.

In [ ]:
from pathlib import Path
import requests

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None

OWNER = "dantasqu"
REPOSITORY = "llm105-allegro-potentials"
REVISION = "main"
RUN_DIR = Path("/content/llm105_allegro_nve/run")
RUN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_REPO_PATH = "models/fine_tuned/finetuned_model_d_omc25_llm105.nequip.pth"
INPUT_REPO_PATH = "examples/nve/in.nve"
STRUCTURE_REPO_PATH = "examples/nve/llm105_uc_std.data"
MODEL_PATH = RUN_DIR / "finetuned_model_d_omc25_llm105.nequip.pth"
INPUT_PATH = RUN_DIR / "in.nve"
STRUCTURE_PATH = RUN_DIR / "llm105_uc_std.data"

def download_repository_file(repository_path, destination, required=True):
    url = f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/contents/{repository_path}"
    headers = {"Accept": "application/vnd.github.raw+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    response = requests.get(url, params={"ref": REVISION}, headers=headers, timeout=120)
    if response.status_code == 404 and not required:
        print(f"Not yet available in the repository: {repository_path}")
        return False
    response.raise_for_status()
    destination.write_bytes(response.content)
    print(f"Downloaded {repository_path} ({destination.stat().st_size:,} bytes)")
    return True

download_repository_file(MODEL_REPO_PATH, MODEL_PATH, required=True)
download_repository_file(INPUT_REPO_PATH, INPUT_PATH, required=True)
download_repository_file(STRUCTURE_REPO_PATH, STRUCTURE_PATH, required=True)

In [ ]:
import hashlib
import zipfile

EXPECTED_SHA256 = {
    MODEL_PATH: "d4103ddd2de524a33dcdef856af64021275e33fc0313072e1ce9c014b9905bc4",
    INPUT_PATH: "4205f6ef3032bd0384e532fb711f6c785bbddf3ed94c88a2e38635b90e1d7c2a",
    STRUCTURE_PATH: "24076d571bd2858578ed0682732777edba2267614626df4424cb8d75a74eea59",
}
for path, expected_digest in EXPECTED_SHA256.items():
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name} SHA-256: {digest}")
    assert digest == expected_digest, f"Checksum mismatch for {path.name}."

with zipfile.ZipFile(MODEL_PATH) as archive:
    names = archive.namelist()
    def read_extra(field):
        member = next(name for name in names if name.endswith(f"/extra/{field}"))
        return archive.read(member).decode().strip()
    metadata = {field: read_extra(field) for field in ["type_names", "num_types", "r_max", "model_dtype", "allow_tf32"]}

print("Embedded model metadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")
assert metadata["type_names"].split() == ["C", "H", "N", "O"]
assert metadata["r_max"] == "5.2"

## 5. Prepare a reviewer-scale version of the verified NVE protocol

The repository preserves the supplied HPC input unchanged. For Colab, this cell changes the `read_data` and deployed-model paths and shortens only the final production command from 10,000,000 to 2,500 steps (1.25 ps at 0.5 fs). It retains the two minimizations, 300 K velocity initialization, 5,000-step gentle start (0.5 ps at 0.1 fs), thermostat-free NVE production stage, output fields, and random seed. The exact change is printed before execution.

In [ ]:
COLAB_INPUT_PATH = RUN_DIR / "in.colab.nve"
DEMO_PRODUCTION_STEPS = 2500
adapted_lines = []
read_data_replaced = False
model_replaced = False
production_timestep_seen = False
production_run_replaced = False
original_production_steps = None

for original_line in INPUT_PATH.read_text().splitlines():
    line = original_line
    stripped = line.strip()
    if stripped.startswith("read_data "):
        indentation = line[:len(line) - len(line.lstrip())]
        line = f"{indentation}read_data {STRUCTURE_PATH}"
        read_data_replaced = True
    elif stripped.startswith("pair_coeff ") and ".nequip.pth" in stripped:
        tokens = stripped.split()
        model_index = next(i for i, token in enumerate(tokens) if token.endswith(".nequip.pth"))
        tokens[model_index] = str(MODEL_PATH)
        line = " ".join(tokens)
        model_replaced = True
    elif stripped.startswith("timestep "):
        production_timestep_seen = float(stripped.split()[1]) == 0.0005
    elif production_timestep_seen and stripped.startswith("run ") and not production_run_replaced:
        indentation = line[:len(line) - len(line.lstrip())]
        original_production_steps = int(stripped.split()[1])
        line = f"{indentation}run {DEMO_PRODUCTION_STEPS}  # Colab reviewer demonstration"
        production_run_replaced = True
    adapted_lines.append(line)

if not read_data_replaced:
    raise ValueError("No read_data command was found in the supplied NVE input.")
if not model_replaced:
    raise ValueError("No deployed .nequip.pth path was found in the supplied pair_coeff command.")
if not production_run_replaced:
    raise ValueError("No production run following the 0.5 fs timestep was found.")

COLAB_INPUT_PATH.write_text("\n".join(adapted_lines) + "\n")
print(f"Reviewer demonstration: production run shortened from {original_production_steps:,} to {DEMO_PRODUCTION_STEPS:,} steps.")
print(f"The original repository input remains unchanged at {INPUT_REPO_PATH}.")
print("NVE input:", INPUT_PATH)
print("Structure:", STRUCTURE_PATH)

protocol_commands = ("units", "atom_style", "boundary", "read_data", "pair_style",
                     "pair_coeff", "min_style", "minimize", "velocity", "timestep",
                     "fix", "run", "thermo", "thermo_style")
for line in adapted_lines:
    if line.strip().startswith(protocol_commands):
        print(line)

## 6. Run the NVE demonstration

The run uses the non-Kokkos `allegro` pair style, matching the verified direct build. LAMMPS output is retained as `validation_success.log` for inspection.

In [ ]:
import time

start = time.perf_counter()
result = subprocess.run(
    [str(LMP), "-in", COLAB_INPUT_PATH.name],
    cwd=RUN_DIR,
    env=runtime_env,
    capture_output=True,
    text=True,
)
elapsed = time.perf_counter() - start
VALIDATION_LOG = RUN_DIR / "validation_success.log"
VALIDATION_LOG.write_text(result.stdout + "\n--- STDERR ---\n" + result.stderr)

print(f"LAMMPS exit code: {result.returncode}")
print(f"Elapsed time: {elapsed / 60:.2f} minutes")
print("Log file:", VALIDATION_LOG)
if result.returncode != 0:
    print(result.stdout[-4000:])
    print(result.stderr[-4000:])
    raise RuntimeError("LAMMPS did not complete successfully. See validation_success.log above.")
print(result.stdout[-3000:])

## 7. Inspect temperature and energy conservation

The parser below finds numeric LAMMPS thermo blocks and plots the final block containing temperature and total energy. The reported drift is a short-run diagnostic, not a substitute for the full stability analysis in the manuscript.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def read_thermo_blocks(log_path):
    blocks = []
    header = None
    rows = []
    for raw_line in Path(log_path).read_text(errors="replace").splitlines():
        fields = raw_line.split()
        if fields and fields[0] == "Step" and "Temp" in fields:
            if header and rows:
                blocks.append(pd.DataFrame(rows, columns=header))
            header, rows = fields, []
            continue
        if header and len(fields) == len(header):
            try:
                rows.append([float(value) for value in fields])
            except ValueError:
                if rows:
                    blocks.append(pd.DataFrame(rows, columns=header))
                header, rows = None, []
    if header and rows:
        blocks.append(pd.DataFrame(rows, columns=header))
    return blocks

blocks = read_thermo_blocks(VALIDATION_LOG)
candidates = [block for block in blocks if len(block) >= 2 and "Temp" in block and "TotEng" in block]
if not candidates:
    raise ValueError("No multi-row thermo block containing Temp and TotEng was found.")
thermo = candidates[-1]

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(thermo["Step"], thermo["Temp"], linewidth=1.2)
axes[0].set_ylabel("Temperature (K)")
axes[0].grid(alpha=0.25)
axes[1].plot(thermo["Step"], thermo["TotEng"], linewidth=1.2)
axes[1].set_xlabel("LAMMPS step")
axes[1].set_ylabel("Total energy (eV)")
axes[1].grid(alpha=0.25)
fig.tight_layout()
plt.show()

n_atoms = 304
energy_drift_per_atom = (thermo["TotEng"].iloc[-1] - thermo["TotEng"].iloc[0]) / n_atoms
print(f"Initial temperature: {thermo['Temp'].iloc[0]:.3f} K")
print(f"Final temperature:   {thermo['Temp'].iloc[-1]:.3f} K")
print(f"Total-energy drift: {energy_drift_per_atom:.6e} eV/atom")

## Interpretation

A successful reviewer check should show that:

- the downloaded model matches the published SHA-256 checksum;
- its embedded atom-type order is `C H N O` and its cutoff is 5.2 Å;
- the compiled LAMMPS executable lists the Allegro pair style;
- minimization, velocity initialization, gentle start, and NVE production complete without a LAMMPS error; and
- temperature and total energy remain finite over the demonstration interval.

The embedded archive metadata identifies the model inference dtype as `float32`. This is distinct from the numerical precision used by other portions of a LAMMPS build. The notebook therefore reports the model dtype directly rather than describing the complete workflow as uniformly single or double precision.